#Baseline LLMA


## Install dependencies

In [14]:
!pip install -U transformers accelerate langchain-huggingface \
    bert-score rouge-score nltk pydantic python-dotenv


## Imports

In [15]:


import os
import json
import torch
import nltk

from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List

from nltk.translate.bleu_score import sentence_bleu, corpus_bleu
from bert_score import score
from rouge_score import rouge_scorer

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM
)

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


## Dataset

In [16]:
class QNAPair(BaseModel):
    id: int
    question: str
    reference_answer: str


class AgriData(BaseModel):
    qa_pairs: List[QNAPair]


with open(
    "/content/drive/MyDrive/Colab Notebooks/Agriculture_research/Bangla_Agriculture_QA_Test_200.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

agri_data = AgriData(**data)



## Hugging Face authentication

In [17]:

load_dotenv()


hf_token = ";)"



## Load Llama 3.2 1B Instruct model

In [18]:


# ROUGE scorer

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False
)

model_id = "meta-llama/Llama-3.2-1B-Instruct"


tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token
)


if torch.cuda.is_available():

    if torch.cuda.is_bf16_supported():
        torch_dtype = torch.bfloat16
    else:
        torch_dtype = torch.float16

else:
    torch_dtype = torch.float32


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True
)


if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id


# Llama 3.x end-of-turn token
eot_token_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")

terminators = [tokenizer.eos_token_id]

if (
    eot_token_id is not None
    and eot_token_id != tokenizer.unk_token_id
):
    terminators.append(eot_token_id)


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

## Hugging Face generation pipeline

In [19]:

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,

    max_new_tokens=100,

    do_sample=False,

    return_full_text=False,

    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=terminators,
)


llm = HuggingFacePipeline(
    pipeline=pipe
)


##Prompt

In [20]:

#ZERO SHOT

# prompt = PromptTemplate(
#     template="""
# নিচের প্রশ্নটির উত্তর বাংলায় দিন।

# প্রশ্নঃ {question}
# উত্তরঃ
# """,
#     input_variables=["question"]
# )


# ONE SHOT

# prompt = PromptTemplate(
#     template="""
# নিচের উদাহরণটি লক্ষ্য করুন এবং একইভাবে বাংলায় উত্তর দিন।

# প্রশ্নঃ টমেটো গাছে পার্শ্বকুশি ও মরা পাতা ছাঁটাই করার সুবিধা কী?
# উত্তরঃ পার্শ্বকুশি ও মরা পাতা ছাঁটাই করলে গাছে রোগ-বালাই ও
# পোকার আক্রমণ কমে এবং ফলের আকার বড় হয়।

# প্রশ্নঃ {question}
# উত্তরঃ
# """,
#     input_variables=["question"]
# )


# FEW SHOT


prompt = PromptTemplate(
    template="""
নিচের উদাহরণগুলো লক্ষ্য করুন এবং একইভাবে বাংলায় উত্তর দিন।

প্রশ্নঃ টমেটোর চারা রোপণের সঠিক দূরত্ব ও বয়স কত?
উত্তরঃ ২৫-৩০ দিন বয়সের টমেটোর চারা প্রতিটি বেডে দুটি
সারিতে ৬০×৪০ সেমি দূরত্বে রোপণ করতে হয়।

প্রশ্নঃ টমেটো চাষে হেক্টরপ্রতি কী পরিমাণ গোবর সার প্রয়োগ করতে হয়?
উত্তরঃ টমেটোর ভালো ফলনের জন্য হেক্টরপ্রতি ৮ থেকে ১২ টন
পচা গোবর সার প্রয়োগ করতে হয়।

প্রশ্নঃ টমেটোর জমিতে ইউরিয়া এবং এমপি সার কখন ও কীভাবে দিতে হয়?
উত্তরঃ পার্শ্বকুশি ছাঁটাইয়ের পর চারা লাগানোর ৩য় ও ৫ম
সপ্তাহে রিং পদ্ধতিতে দুই কিস্তিতে এই সারগুলো দিতে হয়।

প্রশ্নঃ টমেটো গাছে পার্শ্বকুশি ও মরা পাতা ছাঁটাই করার সুবিধা কী?
উত্তরঃ পার্শ্বকুশি ও মরা পাতা ছাঁটাই করলে গাছে রোগ-বালাই ও
পোকার আক্রমণ কমে এবং ফলের আকার বড় হয়।

প্রশ্নঃ টমেটো গাছকে প্রবল বাতাস থেকে রক্ষা করতে কী ব্যবস্থা নেওয়া হয়?
উত্তরঃ টমেটো গাছ যাতে নুয়ে না পড়ে সেজন্য বাঁশের তৈরি কাঠি,
কঞ্চি বা ডাল দিয়ে ‘এ’ আকৃতির ঠেকনা দেওয়া হয়।

প্রশ্নঃ {question}
উত্তরঃ
""",
    input_variables=["question"]
)


# CHAIN-OF-THOUGHT STYLE PROMPT


# prompt = PromptTemplate(
#     template="""
# আপনি একজন কৃষি বিশেষজ্ঞ।

# প্রশ্নটি বুঝে প্রয়োজনীয় তথ্য বিবেচনা করুন।
# সবশেষে শুধু সংক্ষিপ্ত ও সঠিক চূড়ান্ত উত্তর বাংলায় দিন।

# প্রশ্নঃ {question}

# চূড়ান্ত উত্তরঃ
# """,
#     input_variables=["question"]
# )



# INSTRUCTION PROMPT


# prompt = PromptTemplate(
#     template="""
# আপনি একজন কৃষি সহায়ক বাংলা প্রশ্নোত্তর সহকারী।
#
# নির্দেশনা:
# - শুধুমাত্র বাংলায় উত্তর দিন
# - সংক্ষিপ্ত ও নির্ভুল উত্তর দিন
# - অতিরিক্ত ব্যাখ্যা যোগ করবেন না
# - প্রশ্ন অনুযায়ী সরাসরি উত্তর দিন
#
# প্রশ্নঃ {question}
# উত্তরঃ
# """,
#     input_variables=["question"]
# )



## Apply Llama chat template

In [21]:
def format_llama_prompt(inputs):

    user_prompt = prompt.format(
        question=inputs["question"]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "আপনি একজন কৃষি বিষয়ক প্রশ্নোত্তর সহকারী। "
                "আপনাকে বাংলায় সংক্ষিপ্ত ও সঠিক উত্তর দিতে হবে।"
            )
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt


chat_formatter = RunnableLambda(format_llama_prompt)

parser = StrOutputParser()


chain = chat_formatter | llm | parser



## Generate predictions

In [24]:

result = []



for pair in agri_data.qa_pairs:

    predicted = chain.invoke({
        "question": pair.question
    }).strip()

    result.append({
        "id": pair.id,
        "question": pair.question,
        "reference_answer": pair.reference_answer,
        "predicted_answer": predicted
    })

    print("=" * 80)
    print("Question:", pair.question)
    print("Reference:", pair.reference_answer)
    print("Prediction:", predicted)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: চুইঝাল সম্পর্কে গাছের গোড়া বিষয়ে কী বলা হয়েছে?
Reference: সার দেয়ার সময় গাছের গোড়ায় মাটি হালকা করে কুপিয়ে দিতে হয় এবং সেচ প্রদান করতে হবে।
Prediction: প্রশ্নগুলো সঠিকভাবে উত্তর দেওয়া হয়নি। আমি এই প্রশ্নগুলো সঠিকভাবে উত্তর দিত
Question: পানিকচুর বন্যাপ্রবণ জায়গায় চারা লাগানোর সময় কী?
Reference: যে সব জায়গা বন্যার পানিতে তলিয়ে যাবার সম্ভাবনা আছে সেখানে কার্তিক মাসেই চারা লাগানো ভালো।
Prediction: প্রশ্নঃ টমেটোর চাষে হেক্টরপ্রতি কী পরিমাণ গোবর সার প্রয়োগ করতে হয়?

উত্তরঃ টম


## Evaluation

In [25]:
all_ans = []
all_pred = []

avg_bleu = 0
avg_rouge = 0
exact_match_c = 0


for item in result:

    ans_tokens = list(item["reference_answer"])
    pred_tokens = list(item["predicted_answer"])


    #BLEU

    sentence_bleu_score = sentence_bleu(
        [ans_tokens],
        pred_tokens
    )

    item["sentence_bleu"] = sentence_bleu_score


    #ROUGE-L

    rouge_res = rouge.score(
        item["reference_answer"],
        item["predicted_answer"]
    )["rougeL"].fmeasure

    item["rouge_score"] = rouge_res


    #Exact Match

    exact_match = int(
        item["reference_answer"].strip()
        ==
        item["predicted_answer"].strip()
    )

    item["exact_match"] = exact_match


    # Corpus BLEU data

    all_ans.append([ans_tokens])
    all_pred.append(pred_tokens)


    avg_bleu += sentence_bleu_score
    avg_rouge += rouge_res
    exact_match_c += exact_match


# Average BLEU / ROUGE / EM


avg_bleu = avg_bleu / len(result)
avg_rouge = avg_rouge / len(result)
exact_match_c = exact_match_c / len(result)



# BERTScore

preds = [
    item["predicted_answer"]
    for item in result
]

references = [
    item["reference_answer"]
    for item in result
]


precision, recall, f1 = score(
    preds,
    references,
    lang="bn",
    verbose=True
)


avg_bert_score_f1 = f1.mean().item()



for i, item in enumerate(result):

    item["bert_score_precision"] = precision[i].item()
    item["bert_score_recall"] = recall[i].item()
    item["bert_score_f1"] = f1[i].item()


# Corpus BLEU

corpus_bleu_score = corpus_bleu(
    all_ans,
    all_pred
)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.11 seconds, 17.56 sentences/sec


## Save result

In [26]:


output_file = "llama_3_2_1b_instruct_baseline.json"

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\n" + "=" * 60)
print("Llama 3.2 1B Instruct Baseline")
print("=" * 60)

print("Average BLEU Score:", avg_bleu)
print("Average ROUGE-L Score:", avg_rouge)
print("Average BERTScore F1:", avg_bert_score_f1)
print("Exact Match Accuracy:", exact_match_c)
print("Corpus BLEU Score:", corpus_bleu_score)

print("\nResults saved to:", output_file)


Llama 3.2 1B Instruct Baseline
Average BLEU Score: 0.12734514456415127
Average ROUGE-L Score: 0.0
Average BERTScore F1: 0.6326857209205627
Exact Match Accuracy: 0.0
Corpus BLEU Score: 0.1295075870281957

Results saved to: llama_3_2_1b_instruct_baseline.json
